# Prédiction de la consommation de carburant (LSTM — approche TinyML)
## Régression avec un réseau LSTM léger sur le dataset Vehicle Attributes and Emissions

**Objectif du TP :**
- Prédire la **consommation de carburant** (`FUEL CONSUMPTION`, en L/100 km) à partir des attributs du véhicule
- Utiliser un petit réseau **LSTM** entraîné en régression
- Concevoir un modèle très léger (~2 300 paramètres), déployable sur microcontrôleur (**TinyML**)
- Standardiser les entrées uniquement sur l'ensemble d'entraînement

> Ce TP utilise **TensorFlow/Keras** (et non PyTorch comme les TP1/TP2) afin de
> rester fidèle au cadre du professeur et de permettre l'export TinyML.  
> Dataset : *Vehicle Attributes and Emissions* (Kaggle — `krupadharamshi/fuelconsumption`).

## 0. Imports et configuration

In [ ]:
import os

import numpy as np

import pandas as pd

import seaborn as sns

import matplotlib.pyplot as plt



import tensorflow as tf

from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



import warnings

warnings.filterwarnings("ignore")



# ─── Dossier de sauvegarde des figures ────────────────────────────────────────

os.makedirs("figs", exist_ok=True)



# ─── Graines aléatoires pour la reproductibilité ──────────────────────────────

tf.random.set_seed(42)

np.random.seed(42)



print(f"TensorFlow version : {tf.__version__}")

print("✓ Configuration initialisée")

## 1. Chargement des données

### Présentation du jeu de données

| Propriété | Valeur |
|-----------|--------|
| Source | Kaggle — `krupadharamshi/fuelconsumption` |
| Variables prédictives | `ENGINE SIZE`, `CYLINDERS`, `COEMISSIONS` |
| Cible | `FUEL CONSUMPTION` (L/100 km) |
| Type de tâche | Régression supervisée |

Téléchargement automatique via `kagglehub`. En cas d'échec, déposer manuellement `FuelConsumption.csv` dans le répertoire de travail.

In [ ]:
# ─── Recherche du fichier CSV (local → Kaggle → erreur) ───────────────────────

CSV = "FuelConsumption.csv"



def find_csv():

    """Recherche le fichier CSV dans les emplacements courants."""

    for p in [CSV, "data/" + CSV, "/content/" + CSV, "/content/data/" + CSV]:

        if os.path.exists(p):

            return p

    return None



path = find_csv()

if path is None:

    try:

        import kagglehub

        d = kagglehub.dataset_download("krupadharamshi/fuelconsumption")

        for root, _, files in os.walk(d):

            for f in files:

                if f.lower().endswith(".csv"):

                    path = os.path.join(root, f)

    except Exception as e:

        print("kagglehub indisponible :", e)



if path is None:

    raise FileNotFoundError(

        "Déposez FuelConsumption.csv dans le répertoire de travail "

        "(Kaggle : krupadharamshi/fuelconsumption).")



# ─── Lecture et nettoyage initial ─────────────────────────────────────────────

df = pd.read_csv(path)

df.columns = df.columns.str.strip()   # supprime les espaces dans les noms de colonnes

print(f"✓ Fichier chargé : {path}")

print(f"  Dimensions : {df.shape}")

df.head()

## 2. Nettoyage des données

Suppression des valeurs manquantes et des doublons pour garantir la qualité du jeu d'entraînement.

In [ ]:
# ─── Suppression des valeurs manquantes et des doublons ───────────────────────

df = df.dropna().drop_duplicates().reset_index(drop=True)

print(f"✓ Après nettoyage : {df.shape[0]} lignes, {df.shape[1]} colonnes")

df.describe()

## 3. Analyse exploratoire (EDA)

### Variables numériques retenues

| Variable | Description |
|----------|-------------|
| `ENGINE SIZE` | Cylindrée du moteur (litres) |
| `CYLINDERS` | Nombre de cylindres |
| `FUEL CONSUMPTION` | **Cible** — consommation (L/100 km) |
| `COEMISSIONS` | Émissions de CO₂ (g/km) |

On visualise les relations entre ces variables via un **pairplot** et une **matrice de corrélation de Spearman**.

In [ ]:
# ─── Variables numériques d'intérêt ───────────────────────────────────────────

num = ["ENGINE SIZE", "CYLINDERS", "FUEL CONSUMPTION", "COEMISSIONS"]



# ─── Pairplot : relations pairwise ────────────────────────────────────────────

sns.pairplot(df[num])

plt.savefig("figs/tp3_pairplot.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp3_pairplot.png")

In [ ]:
# ─── Matrice de corrélation de Spearman ───────────────────────────────────────

#     Spearman est préférable à Pearson pour des variables non-linéaires

corr = df[num].corr("spearman")

plt.figure(figsize=(8, 6))

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,

            xticklabels=num, yticklabels=num)

plt.title("Matrice de corrélation de Spearman")

plt.xticks(rotation=30, ha="right")

plt.tight_layout()

plt.savefig("figs/tp3_heatmap.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp3_heatmap.png")

## 4. Préparation : variables, normalisation et mise en forme

### Pourquoi standardiser les entrées ?

Les variables d'entrée ont des échelles très différentes (`ENGINE SIZE` ~ 1–8,
`COEMISSIONS` ~ 100–500). Sans normalisation, le gradient sera dominé par
les grandes valeurs et la convergence sera instable.

Le `StandardScaler` est **ajusté uniquement sur l'entraînement** pour éviter
la fuite d'information vers le test.

### Mise en forme pour le LSTM

Le LSTM attend des entrées de forme `(échantillons, timesteps, features)`.  
Ici, on traite chaque variable comme un **timestep** :

```
X.shape : (N, 3, 1)   →   3 timesteps, 1 feature par pas
```

In [ ]:
# ─── Sélection des variables et de la cible ───────────────────────────────────

features = ["ENGINE SIZE", "CYLINDERS", "COEMISSIONS"]

target   = "FUEL CONSUMPTION"



X = df[features].values.astype("float32")

y = df[target].values.astype("float32").reshape(-1, 1)



# ─── Split train / test (70 % / 30 %) ────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.3, random_state=42

)



# ─── Standardisation ajustée sur l'entraînement uniquement ───────────────────

scaler  = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train).astype("float32")

X_test  = scaler.transform(X_test).astype("float32")



# ─── Mise en forme (samples, timesteps=3, features=1) ────────────────────────

#     Chaque variable est traitée comme un pas temporel

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)



print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")

print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")

print("✓ Données prêtes")

## 5. Architecture : LSTM de régression (TinyML)

### Pourquoi un modèle aussi léger ?

L'objectif **TinyML** est de déployer le modèle sur un microcontrôleur
(Arduino, STM32, etc.) dont la RAM est de l'ordre de quelques dizaines de ko.
Le modèle doit donc être minuscule.

### Architecture retenue

```
Entrée (3 timesteps × 1 feature)
       ↓
 [LSTM  12 unités, return_sequences=True ]
       ↓
 [LSTM  12 unités                        ]
       ↓
 [Dense 32 unités, activation ReLU       ]
       ↓
 [Dense  1 unité,  activation linéaire   ]  ← prédiction (L/100 km)
```

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| Unités LSTM | 12 | Modèle léger pour TinyML |
| Optimiseur | Adamax | Robuste sur données réelles |
| Perte | MAE | Interprétable en L/100 km |

In [ ]:
# ─── Construction du modèle LSTM ──────────────────────────────────────────────

def build_lstm(input_shape=(3, 1)):

    """

    Construit un LSTM de régression léger pour le contexte TinyML.



    Architecture : LSTM(12) → LSTM(12) → Dense(32, relu) → Dense(1, linear)

    Paramètres : ~2 300  (déployable sur microcontrôleur)

    """

    model = tf.keras.Sequential([

        layers.Input(shape=input_shape),

        layers.LSTM(12, return_sequences=True),   # 1re couche LSTM — retourne toute la séquence

        layers.LSTM(12),                          # 2e couche LSTM — retourne uniquement le dernier état

        layers.Dense(32, activation="relu"),      # couche dense intermédiaire

        layers.Dense(1,  activation="linear"),    # sortie scalaire (consommation en L/100 km)

    ])

    model.compile(optimizer="adamax", loss="mae", metrics=["mae"])

    return model



# ─── Instanciation et résumé ──────────────────────────────────────────────────

model = build_lstm((X_train.shape[1], 1))

model.summary()

print(f"\n✓ Modèle construit — {model.count_params():,} paramètres")

## 6. Entraînement

300 époques avec 20 % de validation interne. Le paramètre `verbose=0` supprime l'affichage ligne par ligne ; on affiche uniquement les métriques finales.

In [ ]:
# ─── Entraînement du modèle ───────────────────────────────────────────────────

history = model.fit(

    X_train, y_train,

    validation_split=0.2,   # 20 % des données d'entraînement pour la validation

    epochs=300,

    batch_size=32,

    verbose=0               # silencieux — on affiche uniquement le résumé final

)

print("✓ Entraînement terminé")

print(f"  MAE finale → train : {history.history['mae'][-1]:.3f}  "

      f"| val : {history.history['val_mae'][-1]:.3f}")

In [ ]:
# ─── Courbe de perte Train / Validation ───────────────────────────────────────

plt.figure(figsize=(9, 5))

plt.plot(history.history["loss"],     label="Train")

plt.plot(history.history["val_loss"], label="Validation")

plt.title("LSTM de régression — Courbe de perte MAE")

plt.xlabel("Époque")

plt.ylabel("MAE (L/100 km)")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig("figs/tp3_loss.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp3_loss.png")

## 7. Évaluation des performances

### Métriques utilisées

| Métrique | Description |
|----------|-------------|
| MAE | Erreur absolue moyenne (L/100 km) |
| RMSE | Racine de l'erreur quadratique moyenne |
| R² | Coefficient de détermination (1 = prédiction parfaite) |

In [ ]:
# ─── Génération des prédictions sur le test ───────────────────────────────────

y_pred = model.predict(X_test, verbose=0)



# ─── Calcul des métriques ─────────────────────────────────────────────────────

mae  = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2   = r2_score(y_test, y_pred)



print("─" * 40)

print(f"  MAE  : {mae:.4f} L/100 km")

print(f"  RMSE : {rmse:.4f} L/100 km")

print(f"  R²   : {r2:.4f}")

print("─" * 40)



# ─── Sauvegarde des résultats ─────────────────────────────────────────────────

pd.DataFrame({"MAE": [mae], "RMSE": [rmse], "R2": [r2]}).to_csv(

    "figs/tp3_results.csv", index=False

)

print("✓ Résultats sauvegardés : figs/tp3_results.csv")

In [ ]:
# ─── Visualisation : valeurs réelles vs prédites ──────────────────────────────

plt.figure(figsize=(12, 5))

plt.plot(y_test[:120], label="Réel",   marker=".")

plt.plot(y_pred[:120], label="Prédit", marker=".", alpha=0.8)

plt.title("Consommation réelle vs prédite (ensemble de test)")

plt.xlabel("Échantillon")

plt.ylabel("FUEL CONSUMPTION (L/100 km)")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig("figs/tp3_pred.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp3_pred.png")

In [ ]:
# ─── Analyse des résidus ──────────────────────────────────────────────────────

#     Résidu = réel − prédit ; on s'attend à une distribution centrée en 0

residuals = y_test.ravel() - y_pred.ravel()

m, s = residuals.mean(), residuals.std()



fig, ax = plt.subplots(1, 2, figsize=(13, 5))



# Résidus vs valeurs prédites

ax[0].scatter(y_pred.ravel(), residuals, alpha=0.6)

ax[0].axhline(0,       color="r",          label="zéro")

ax[0].axhline(m + 2*s, color="g", ls="--", label="±2σ")

ax[0].axhline(m - 2*s, color="g", ls="--")

ax[0].set_title("Résidus vs prédictions")

ax[0].set_xlabel("Valeur prédite")

ax[0].set_ylabel("Résidu")

ax[0].legend()

ax[0].grid(True)



# Distribution des résidus

ax[1].hist(residuals, bins=20, color="steelblue", alpha=0.7)

ax[1].axvline(m, color="k", ls="--", label=f"moyenne = {m:.3f}")

ax[1].set_title("Distribution des résidus")

ax[1].set_xlabel("Résidu")

ax[1].legend()

ax[1].grid(True)



plt.tight_layout()

plt.savefig("figs/tp3_residuals.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp3_residuals.png")

## 8. Vers le TinyML — export pour microcontrôleur (bonus)

### Pourquoi ce modèle est-il TinyML-compatible ?

| Critère | Valeur |
|---------|--------|
| Paramètres | ~2 300 |
| Taille en mémoire | ~9 ko |
| Framework d'export | TensorFlow Lite |
| Générateur de code C | `eloquent_tensorflow` |

La bibliothèque `eloquent_tensorflow` génère directement le code C embarquable.
Cette étape est facultative — elle peut être ignorée si la bibliothèque n'est pas disponible.

In [ ]:
# ─── Export TinyML (optionnel) ────────────────────────────────────────────────

try:

    import subprocess, sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eloquent-tensorflow"])

    from eloquent_tensorflow import convert_model



    c_code = convert_model(model)

    print(c_code[:600])   # aperçu de l'en-tête C généré



    with open("model_tinyml.h", "w") as f:

        f.write(c_code)

    print("\n✓ Modèle C exporté → model_tinyml.h")

except Exception as e:

    print("Export TinyML ignoré :", e)